# A small CNN

Fashion-MNIST is 70,000 clothing photos, 28 by 28, grayscale, ten classes. The files come from the Zalando research repo and are not in git. If `data/fashion/` is missing, this notebook uses scikit-learn digits instead. Digits are 8 by 8.

The network is two convolution layers and two pooling layers, then one linear layer. It trains on CPU. Fashion-MNIST uses the full seeded split: 42,000 train rows, 14,000 validation rows, and 14,000 held-out rows, for 12 epochs. Digits use every row for 40 epochs.

The clothing classes are a stand-in for a product photo. They are not groceries.


In [ ]:
import os
import sys
from pathlib import Path

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    for folder in (candidate, candidate / "04-deep-learning"):
        marker = folder / "lib" / "customers.py"
        if marker.is_file() and (folder / "app.py").is_file():
            root = folder
            break
    if root is not None:
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from the repo folder or from 04-deep-learning.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
from cnn import class_epochs, confusion, predict, save_cnn, train_cnn
from cup import class_parts, prediction_frame, write_frame

bundle, train, val, hold_x = class_parts("auto")
print("source", bundle["source"], "image", bundle["image_size"], "classes", len(bundle["classes"]))
print("train", len(train[0]), "validation", len(val[0]), "held-out", len(hold_x))


## Train

Exercise. `learning_rate` starts at 0.001. Try 0.01 after the first fit and compare the validation accuracy.


In [ ]:
learning_rate = 0.001  # Exercise. Try 0.01, then run this cell again.

result = train_cnn(
    train[0],
    train[1],
    val[0],
    val[1],
    bundle["image_size"],
    len(bundle["classes"]),
    learning_rate=learning_rate,
    epochs=class_epochs(bundle["source"]),
)
print("train accuracy", round(result["train_accuracy"], 3))
print("validation accuracy", round(result["validation_accuracy"], 3))
print("seconds", round(result["seconds"], 2))
save_cnn(ROOT / "models" / "cnn.pt", result["model"], bundle["image_size"], bundle["classes"], bundle["source"])
guessed = predict(result["model"], hold_x)
write_frame(prediction_frame(guessed), ROOT / "models" / "cup_predictions.csv")
print("wrote models/cup_predictions.csv")
matrix = confusion(result["model"], val[0], val[1], len(bundle["classes"]))
print(matrix)


The held-out labels are not in this notebook. From this folder, the teaching team scores the file:

```sh
python score_cup.py models/cup_predictions.csv path/to/cup_labels.csv
```

## Debrief

Exercise. One sentence: did a higher learning rate help on the validation rows?


In [ ]:
debrief = ""  # Exercise. One sentence.
print(debrief or "Write one sentence, then run this cell again.")
